# WideBind — Colab Training (D=3584, T4)
165M params, 24 layers, grouped MLP, cognitive mirror, adaptive gates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/widebind_data'
print(f'Data path: {DRIVE}')
!ls -lh "$DRIVE"/*.bin 2>/dev/null || echo 'No .bin files found'

In [ ]:
# Locate source files on Drive and copy locally
import sys, os, shutil
SRC_LOCAL = '/content/widebind_src'
NEEDED = ['core.py', 'wbconfig.py', 'analyze_checkpoint.py', 'fcf_cpr.py']

src_drive = None
for d in [os.path.join(DRIVE, 'src'), DRIVE]:
    if all(os.path.isfile(os.path.join(d, f)) for f in NEEDED):
        src_drive = d
        break

if src_drive:
    os.makedirs(SRC_LOCAL, exist_ok=True)
    for f in NEEDED:
        shutil.copy2(os.path.join(src_drive, f), os.path.join(SRC_LOCAL, f))
    print(f'Source files copied from {src_drive}')
    sys.path.insert(0, SRC_LOCAL)
else:
    for d in [os.path.join(DRIVE, 'src'), DRIVE]:
        missing = [f for f in NEEDED if not os.path.isfile(os.path.join(d, f))]
        found = [f for f in NEEDED if os.path.isfile(os.path.join(d, f))]
        print(f'{d}: found {found}, missing {missing}')
    print('\nUpload all 3 files and restart Runtime \u2192 Run all')

%cd /content

In [ ]:
# Verify imports
from wbconfig import WideBindConfig
from core import WideBindStack, MirrorLRScheduler
from fcf_cpr import FCF_CPR

cfg = WideBindConfig(D=3584, n_layers=24, bottleneck=3584, bind_K=16,
                     mlp_groups=32, mlp_expand=8)
model = WideBindStack(cfg)
print(f'Model: {model.param_count():,} params')
del model

In [ ]:
# ─── Launch training ───
# Config
D            = 3584
N_LAYERS     = 24
BIND_K       = 16
MLP_GROUPS   = 32
MLP_EXPAND   = 8
SEQ_LEN      = 128
LR           = 3e-4
MAX_STEPS    = 300000
WARMUP       = 2000
SAVE_EVERY   = 5000
EVAL_EVERY   = 1000
LOG_EVERY    = 100
SCHEDULER    = 'mirror'  # 'mirror' or 'cosine'

print('=' * 60)
print(f'WideBind D={D} G={MLP_GROUPS}x{MLP_EXPAND}×')
print(f'  Layers: {N_LAYERS} | K={BIND_K} | L={SEQ_LEN}')
print(f'  LR={LR} | warmup={WARMUP} | steps={MAX_STEPS}')
print(f'  Scheduler: {SCHEDULER}')
print(f'  Save dir: {os.path.join(DRIVE, "checkpoints")}')
print('=' * 60)

In [ ]:
# ─── Training loop ───
import os, math, sys, time, glob
import torch
import torch.nn.functional as F
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')
mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {mem_total:.1f} GB')

# --- config ---
cfg = WideBindConfig(
    D=D, n_layers=N_LAYERS, bottleneck=D, bind_K=BIND_K,
    mlp_groups=MLP_GROUPS, mlp_expand=MLP_EXPAND,
    seq_len=SEQ_LEN, lr=LR, max_steps=MAX_STEPS,
    warmup_steps=WARMUP, scheduler=SCHEDULER,
    data_dir=DRIVE, save_dir=os.path.join(DRIVE, 'checkpoints'),
    log_dir=os.path.join(DRIVE, 'logs'),
    grad_clip=1.0, conv_kernel=48,
    log_interval=LOG_EVERY, eval_interval=EVAL_EVERY,
    save_interval=SAVE_EVERY,
)

# --- data ---
stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*.bin')))
print(f'Data files: {len(stream_files)}')

# --- memory-efficient data loading (uint16 numpy, converted per-batch) ---
class TokenStream:
    def __init__(self, path):
        self.data = np.fromfile(path, dtype=np.uint16)
        self.len = len(self.data)
    def get_batch(self, seq_len, batch_size, offset):
        needed = batch_size * seq_len + 1
        if offset + needed > self.len:
            offset = 0
        chunk = self.data[offset:offset + needed]
        x = torch.from_numpy(chunk[:batch_size * seq_len].reshape(batch_size, seq_len).copy())
        y = torch.from_numpy(chunk[1:batch_size * seq_len + 1].reshape(batch_size, seq_len).copy())
        return x.long(), y.long(), offset + batch_size * seq_len

streams = [TokenStream(f) for f in stream_files]
total_tokens = sum(s.len for s in streams)
print(f'Total tokens: {total_tokens:,}')
print(f'RAM for data: {total_tokens * 2 / 1e9:.1f} GB (uint16)')

# --- model ---
model = WideBindStack(cfg).to(device)
print(f'Params: {model.param_count():,}')

# --- auto batch ---
bs = 8
torch.cuda.empty_cache()
try:
    x = torch.randint(0, 50000, (8, SEQ_LEN), device=device)
    h = model.embed_tokens(x)
    out, _ = model(h, None)
    out[:, :1].sum().backward()
    bs = 8
except RuntimeError:
    bs = 4
    try:
        x = torch.randint(0, 50000, (4, SEQ_LEN), device=device)
        h = model.embed_tokens(x)
        out, _ = model(h, None)
        out[:, :1].sum().backward()
        bs = 4
    except RuntimeError:
        bs = 2
model.zero_grad()
torch.cuda.empty_cache()
cfg.batch_size = bs
print(f'Batch size: {bs}')

# --- compressor + optimizer + scheduler ---
cpr = FCF_CPR()
optimizer = torch.optim.AdamW(model.param_groups(), betas=(0.9, 0.95))
scheduler = MirrorLRScheduler(model, optimizer, cfg.lr, warmup=cfg.warmup_steps)

# --- resume ---
start_step = 0
best_val_loss = float('inf')
os.makedirs(cfg.save_dir, exist_ok=True)
def step_key(p):
    return int(os.path.basename(p).split('_')[-1].split('.')[0])
ckpt_files = sorted(glob.glob(os.path.join(cfg.save_dir, 'step_*.pt')), key=step_key)
if ckpt_files:
    latest = ckpt_files[-1]
    print(f'Resuming from {latest}')
    ckpt = cpr.load_compressed(latest, cfg=cfg)
    print(f'  Keys in checkpoint: {list(ckpt.keys())}')
    missing, unexpected = model.load_state_dict(ckpt['model'], strict=False)
    if missing: print(f'  Missing keys: {len(missing)}')
    if unexpected: print(f'  Unexpected keys: {len(unexpected)}')
    if 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    else:
        print('  WARNING: no optimizer in checkpoint, starting fresh')
    if 'scheduler' in ckpt:
        try:
            scheduler.load_state_dict(ckpt['scheduler'])
        except Exception as e:
            print(f'  WARNING: scheduler load failed ({e}), using step')
            scheduler._step = ckpt.get('step', 0)
    else:
        scheduler._step = ckpt.get('step', 0)
    start_step = ckpt['step']
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resume at step {start_step}')

# --- helpers ---
@torch.no_grad()
def evaluate_model(model, streams, cfg, device):
    total_loss = 0.0
    total_steps = 0
    n_eval = min(100, sum(s.len for s in streams) // (cfg.batch_size * cfg.seq_len) // len(streams))
    n_warmup = 10
    for stream in streams:
        offset = max(stream.len // 4, cfg.batch_size * cfg.seq_len + 1)
        state = None
        # Warm-up: let state + controller stabilize before measuring
        for i in range(n_warmup):
            x, y, offset = stream.get_batch(cfg.seq_len, cfg.batch_size, offset)
            if offset == 0: break
            x, y = x.to(device), y.to(device)
            h = model.embed_tokens(x)
            out, state = model(h, state)
        # Measure
        for i in range(n_eval):
            x, y, offset = stream.get_batch(cfg.seq_len, cfg.batch_size, offset)
            if offset == 0: break
            x, y = x.to(device), y.to(device)
            h = model.embed_tokens(x)
            out, state = model(h, state)
            loss = model.compute_loss(out, y)
            total_loss += loss.item()
            total_steps += 1
            if i % 50 == 49:
                state = None
    return total_loss / max(total_steps, 1)

# --- train loop ---
state = None
stream_idx = 0
offset = 0
tokens_seen = 0
t0 = time.time()

print(f'Training: step {start_step} → {MAX_STEPS}')
try:
    for step in range(start_step, MAX_STEPS):
        model.train()
        stream = streams[stream_idx]
        x, y, offset = stream.get_batch(SEQ_LEN, bs, offset)
        if offset == 0:
            stream_idx = (stream_idx + 1) % len(streams)
            stream = streams[stream_idx]
            _, _, offset = stream.get_batch(SEQ_LEN, bs, 0)
            state = None
        x, y = x.to(device), y.to(device)

        h = model.embed_tokens(x)
        out, state = model(h, state)
        loss = model.compute_loss(out, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        tokens_seen += bs * SEQ_LEN
        current_lr = scheduler.get_last_lr()[0]

        if step % LOG_EVERY == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(dt, 1e-8)
            mem_u = torch.cuda.max_memory_allocated() / 1e9
            print(f'  step={step:>6} loss={loss.item():.4f} lr={current_lr:.2e} '
                  f'tok/s={tok_s:.0f} vram={mem_u:.1f}/{mem_total:.0f} GB')
            torch.cuda.reset_peak_memory_stats()

        if step > 0 and step % EVAL_EVERY == 0:
            val_loss = evaluate_model(model, streams, cfg, device)
            print(f'  EVAL step={step}: val_loss={val_loss:.4f} '
                  f'val_ppl={math.exp(val_loss):.2f}')
            torch.cuda.empty_cache()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                cpr.save_compressed({
                    'step': step, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_val_loss': best_val_loss, 'cfg': cfg,
                }, os.path.join(cfg.save_dir, 'best.pt'))
                print(f'  New best! Saved to best.pt (compressed)')

        if step > 0 and step % SAVE_EVERY == 0:
            save_path = os.path.join(cfg.save_dir, f'step_{step}.pt')
            cpr.save_compressed({
                'step': step, 'model': model.state_dict(),
                'best_val_loss': best_val_loss, 'cfg': cfg,
            }, save_path)
            print(f'  Checkpoint saved: step_{step}.pt (compressed)')

except KeyboardInterrupt:
    save_path = os.path.join(cfg.save_dir, f'interrupt_step_{step}.pt')
    cpr.save_compressed({
        'step': step, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss, 'cfg': cfg,
    }, save_path)
    print(f'\nInterrupted. Saved to {save_path} (compressed)')

In [ ]:
# ─── Analyze latest checkpoint ───
from analyze_checkpoint import generate_report
import glob

latest_ckpt = sorted(glob.glob(os.path.join(cfg.save_dir, 'step_*.pt')))
if latest_ckpt:
    report_path = generate_report(latest_ckpt[-1])
    from IPython.display import HTML, display
    with open(report_path, 'r') as f:
        display(HTML(f.read()))
else:
    print('No checkpoints found')